<a href="https://colab.research.google.com/github/affe12349/resnet18-cifar10/blob/main/resnet18_data_size.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from torch.utils.data import DataLoader, Subset

import numpy as np
import matplotlib.pyplot as plt
import random


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


cuda


In [ ]:
# Task 1: Hyperparameters and Dataset
num_epochs = 50
learning_rate = 0.001
batch_size = 128

normalize = transforms.Normalize(
    mean=[0.4914, 0.4822, 0.4465],
    std=[0.2470, 0.2435, 0.2616]
)

train_transform = transforms.Compose([
    transforms.Pad(4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32),
    transforms.ToTensor(),
    normalize
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    transform=train_transform,
    download=True
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    transform=test_transform,
    download=True
)


100%|██████████| 170M/170M [00:03<00:00, 47.5MB/s]


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [ ]:
# Load ResNet-18
model = models.resnet18(weights=None)

# Adapt the final layer for CIFAR-10 (10 classes)
model.fc = nn.Linear(model.fc.in_features, 10)

# Move model to the selected device
model = model.to(device)


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    model.train()

    for epoch in range(num_epochs):
        running_loss = 0.0

        # Training loop
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = correct / total
        model.train()

    return val_accuracy


In [ ]:
# Percentages of training data to use
data_percentages = [0.1, 0.25, 0.5, 1.0]

# Dictionary to store train loaders for each percentage
train_loaders = {}

num_train_samples = len(train_dataset)

for p in data_percentages:
    subset_size = int(p * num_train_samples)
    indices = list(range(subset_size))

    subset = Subset(train_dataset, indices)

    train_loaders[p] = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=True
    )


In [ ]:
# Dictionary to store results
results = {}

# Loop over different data percentages
for p, train_loader_subset in train_loaders.items():
    accuracies = []

    # Repeat experiment multiple times for stability
    for run in range(3):
        # Initialize a fresh model for each run
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 10)
        model = model.to(device)

        # Define loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        # Train the model
        val_accuracy = train_model(
            model,
            train_loader_subset,
            test_loader,
            criterion,
            optimizer,
            num_epochs
        )

        accuracies.append(val_accuracy)

    # Store mean accuracy for this data percentage
    results[p] = sum(accuracies) / len(accuracies)

    print(f"Training data: {int(p*100)}% | Validation accuracy: {results[p]:.4f}")


Training data: 10% | Validation accuracy: 0.6415
Training data: 25% | Validation accuracy: 0.7361
Training data: 50% | Validation accuracy: 0.7938
Training data: 100% | Validation accuracy: 0.8429


In [ ]:
sizes = [10, 25, 50, 100]
accs = [results[p] * 100 for p in data_percentages]

plt.figure(figsize=(7, 4.5))
plt.plot(sizes, accs, marker="o")
for x, y in zip(sizes, accs):
    plt.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
plt.xlabel("Training data used (%)")
plt.ylabel("Test accuracy (%)")
plt.title("ResNet-18 on CIFAR-10: accuracy vs training data")
plt.grid(alpha=0.3)
plt.savefig("accuracy_vs_data.png", dpi=150, bbox_inches="tight")
plt.show()